## K-Nearest Neighbors (KNN)

O [KNN](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) é um algoritmo baseado em instâncias, que classifica uma nova observação com base nos seus vizinhos mais próximos.

A classe é definida pela maioria dos vizinhos.

### Hiperparâmetros utilizados

- **n_neighbors**: número de vizinhos considerados (default = 5)
  - Valores menores → modelo mais sensível 
  - Valores maiores → modelo mais estável

- **weights**: forma de ponderação
  - `uniform`: todos os vizinhos têm o mesmo peso (uniform)
  - `distance`: vizinhos mais próximos têm maior peso

### Estratégia

Foi utilizado GridSearchCV com validação cruzada para encontrar os melhores parâmetros, utilizando ROC AUC como métrica.

In [1]:
import pandas as pd
import sys
import os

# add raiz do projeto
sys.path.append(os.path.abspath(".."))

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from preprocessing.main_preprocessing import load_and_preprocess
from utils.experiment_logger import save_results


## BASELINE

In [2]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

smote_options = [False, True]

results = []

for scenario in scenarios:
    for use_smote in smote_options:

        print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

        X_train, X_test, y_train, y_test = load_and_preprocess(
            "../predictive_models/scrdata_202505.csv",
            scenario=scenario,
            use_smote=use_smote
        )

        steps = [
            ('scaler', StandardScaler()),
            ('knn', KNeighborsClassifier(n_neighbors=5)
                    ]

        if use_smote:
            steps.insert(1 if not steps or steps[0][0] != 'scaler' else 1, ('smote', SMOTE(random_state=42)))

        model = Pipeline(steps)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results.append({
            "model": "KNN",
            "scenario": scenario,
            "smote": use_smote,
            "roc_auc": roc_auc_score(y_test, y_proba),
            "f1": f1_score(y_test, y_pred),
            "accuracy": accuracy_score(y_test, y_pred),
            "n_features": X_train.shape[1],
            "phase": "baseline",
            "timestamp": pd.Timestamp.now()
        })

save_results(results, file_path="../results/experiments.csv")

df_result = pd.DataFrame(results)

display(df_result)



🔎 Scenario: sem_submodalidade | SMOTE: False

🔎 Scenario: sem_submodalidade | SMOTE: True

🔎 Scenario: submodalidade_agrupada | SMOTE: False

🔎 Scenario: submodalidade_agrupada | SMOTE: True

🔎 Scenario: submodalidade_engineered | SMOTE: False

🔎 Scenario: submodalidade_engineered | SMOTE: True


,model,scenario,smote,roc_auc,f1,accuracy,n_features,phase,timestamp
0,KNN,sem_submodalidade,False,0.810879,0.778264,0.739332,67,baseline,2026-04-09 21:25:45.516149
1,KNN,sem_submodalidade,True,0.805662,0.761729,0.734331,67,baseline,2026-04-09 21:26:24.483018
2,KNN,submodalidade_agrupada,False,0.811218,0.778614,0.739715,97,baseline,2026-04-09 21:26:58.287996
3,KNN,submodalidade_agrupada,True,0.806085,0.762209,0.734806,97,baseline,2026-04-09 21:27:43.506416
4,KNN,submodalidade_engineered,False,0.810879,0.778264,0.739332,67,baseline,2026-04-09 21:28:11.928374
5,KNN,submodalidade_engineered,True,0.805662,0.761729,0.734331,67,baseline,2026-04-09 21:28:48.995776


## GRIDSEARCH

In [3]:
X_train, X_test, y_train, y_test = load_and_preprocess(
    "../predictive_models/scrdata_202505.csv",
    scenario="sem_submodalidade",
    use_smote=False
)


param_grid_knn = {
    "smote": [SMOTE(random_state=42), "passthrough"],
    "knn__n_neighbors": [3, 5, 7, 9],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"]
}

grid_knn = GridSearchCV(
    estimator=Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('knn', KNeighborsClassifier())
    ]),
    param_grid=param_grid_knn,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_knn.fit(X_train, y_train)

print("Best parameters:", grid_knn.best_params_)
print("Best ROC AUC (CV):", grid_knn.best_score_)


#? BEST MODEL TEST EVALUATION

best_knn = grid_knn.best_estimator_

y_pred = best_knn.predict(X_test)
y_proba = best_knn.predict_proba(X_test)[:, 1]


#? TUNING (CV)

tuning_result = [{
    "model": "KNN",
    "scenario": "sem_submodalidade",
    "smote": False,
    "phase": "tuning_cv",
    "roc_auc": grid_knn.best_score_,
    "f1": None,
    "accuracy": None,
    "best_params": str(grid_knn.best_params_),
    "timestamp": pd.Timestamp.now()
}]

final_result = [{
    "model": "KNN",
    "scenario": "sem_submodalidade",
    "smote": False,
    "phase": "test",
    "roc_auc": roc_auc_score(y_test, y_proba),
    "f1": f1_score(y_test, y_pred),
    "accuracy": accuracy_score(y_test, y_pred),
    "best_params": str(grid_knn.best_params_),
    "timestamp": pd.Timestamp.now()
}]


save_results(tuning_result, file_path="../results/model_results.csv")
save_results(final_result, file_path="../results/model_results.csv")


df_result = pd.DataFrame(final_result)
display(df_result)


Best parameters: {'metric': 'manhattan', 'n_neighbors': 9, 'weights': 'distance'}
Best ROC AUC (CV): 0.8289392016709037


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,KNN,sem_submodalidade,False,test,0.82898,0.782287,0.746322,"{'metric': 'manhattan', 'n_neighbors': 9, 'wei...",2026-04-09 21:46:50.358381
